In [1]:
import sys
sys.path.append('../..')

import yaml
import json
from pathlib import Path
from src.pipeline.med_rag import MedicalRAGPipeline
from evaluation.evaluation_QA_system.RAG_evaluator import RAGEvaluator

## Load Configuration

In [2]:
config_path = '../../configs/pipeline_config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

print("Configuration loaded")

Configuration loaded


## Initialize Pipeline

In [3]:
pipeline = MedicalRAGPipeline(config)
print("Pipeline initialized")

Hint: For standard spaCy, install a model, e.g. 'python -m spacy download en_core_web_sm', and set model_name to 'en_core_web_sm'.


/home/jgibson2/miniconda3/envs/medrag/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/jgibson2/miniconda3/envs/medrag/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
[ENCODER] Using device: cuda
[ENCODER] Loading MedCPT model: ncbi/MedCPT-Query-Encoder on cuda
[ENCODER] Loading tokenizer...
[ENCODER] Loading model...


Loaded Hugging Face NER model: d4data/biomedical-ner-all


[ENCODER] Moving model to cuda...
[ENCODER] Model loaded successfully


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at pritamdeka/S-PubMedBert-MS-MARCO and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[OPENAI] Detected CMU AI Gateway, normalized URL: https://ai-gateway.andrew.cmu.edu/v1
[OPENAI] Initializing with:
  - Model: gpt-4o-mini-2024-07-18
  - Base URL: https://ai-gateway.andrew.cmu.edu/v1
  - Project ID: proj_DMiMpLLTUcyisZg403bSRcDy
[OPENAI] Failed to initialize: OpenAI.__init__() got an unexpected keyword argument 'project'


TypeError: OpenAI.__init__() got an unexpected keyword argument 'project'

## Load Test Queries

In [ ]:
# Load test queries (example format)
test_queries = [
    {
        "query": "What are the symptoms of COVID-19?",
        "relevant_docs": ["doc1", "doc2"],
        "reference_answer": "COVID-19 symptoms include fever, cough, and fatigue."
    },
    # Add more test queries here
]

print(f"Loaded {len(test_queries)} test queries")

Loaded 1 test queries


## Run Evaluation

In [ ]:
evaluator = RAGEvaluator()
predictions = []

for test_query in test_queries:
    result = pipeline.process_query(test_query["query"])
    predictions.append(result)

print(f"Processed {len(predictions)} queries")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Processed 1 queries


## Compute Metrics

In [ ]:
metrics = evaluator.evaluate_batch(predictions, test_queries)

print("\nEvaluation Metrics:")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")


Evaluation Metrics:
avg_recall@5: 0.0000
avg_recall@10: 0.0000
avg_recall@20: 0.0000
avg_mrr: 0.0000
avg_precision@5: 0.0000
avg_precision@10: 0.0000
avg_precision@20: 0.0000


## Save Results

In [ ]:
results_path = '../evaluation_data_storages/evaluation_results.json'
with open(results_path, 'w') as f:
    json.dump({
        'metrics': metrics,
        'predictions': predictions
    }, f, indent=2)

print(f"Results saved to {results_path}")